In [ ]:
!uv pip install -U requests python-dotenv

Using Python 3.12.11 environment at: /Users/yang/developer/pymatgen-2-paper/.venv
Resolved 6 packages in 146ms                                         
Prepared 1 package in 1ms                                                
Installed 1 package in 5ms1                                 
 + python-dotenv==1.1.1


In [ ]:
# Step 1: Get all PRs (title, date)
# Please add `GITHUB_TOKEN` to `.env` at the root of this repo

import os
import requests
import json
from pathlib import Path

from dotenv import load_dotenv

env_path = Path().resolve() / "../../.env"
load_dotenv(dotenv_path=env_path)

url = "https://api.github.com/repos/materialsproject/pymatgen/pulls"
params = {"state": "all", "per_page": 100, "page": 1}

# Setup GitHub token (for higher API rate limit)
if not (token := os.environ.get("GITHUB_TOKEN")):
    raise RuntimeError("Please set your GitHub token in GITHUB_TOKEN env var.")
headers = {"Authorization": f"Bearer {token}"}

# Load previous cache if exists
if os.path.exists("prs.json"):
    with open("prs.json", "r") as f:
        pr_dict = json.load(f)
    print(f"Cache found: {len(pr_dict)} PRs (no fetch).")
else:
    pr_dict = {}  # {pr_number: {title, created_at, author}}

    while True:
        response = requests.get(url, headers=headers, params=params, timeout=60)
        response.raise_for_status()
        prs = response.json()
        if not prs:
            break
        for pr in prs:
            # Only keep merged PRs (comment out to keep all)
            if pr["merged_at"] is None:
                continue

            pr_dict[str(pr["number"])] = {
                "title": pr["title"],
                "created_at": pr["created_at"],
                "author": pr["user"]["login"],
            }
        params["page"] += 1

    with open("prs.json", "w") as f:
        json.dump(pr_dict, f, indent=2)

    print(f"Saved {len(pr_dict)} PRs to prs.json")

Saved 2591 PRs to prs.json


In [ ]:
# Quickly clean up PRs (drop bot PRs and so on)
